# PySpark MLlib Tutorial

#### Example - logistic Regression

In [27]:
import findspark
findspark.init()
import pyspark
import os
import sys
import pyspark.sql.functions as F
import matplotlib.pyplot as plt
plt.style.use(style='seaborn')

from pyspark.ml.feature import VectorAssembler
from pyspark.sql import types as T
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('MLlib Tutorial').getOrCreate()
spark

In [28]:
# read a sample dataset
df = spark.read.csv('SyntheticFinData.csv', header = True, inferSchema = True)

df.show()
df.printSchema()

+----+--------+---------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|step|    type|   amount|   nameOrig|oldbalanceOrg|newbalanceOrig|   nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+---------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|   1| PAYMENT|  9839.64|C1231006815|     170136.0|     160296.36|M1979787155|           0.0|           0.0|      0|             0|
|   1| PAYMENT|  1864.28|C1666544295|      21249.0|      19384.72|M2044282225|           0.0|           0.0|      0|             0|
|   1|TRANSFER|    181.0|C1305486145|        181.0|           0.0| C553264065|           0.0|           0.0|      1|             0|
|   1|CASH_OUT|    181.0| C840083671|        181.0|           0.0|  C38997010|       21182.0|           0.0|      1|             0|
|   1| PAYMENT| 11668.14|C2048537720|      41554.0|      29885.86|M123070170

In [5]:
df1 = df.select('type','amount','oldbalanceOrg','newbalanceOrig','isFraud')
df1.show()

+--------+---------+-------------+--------------+-------+
|    type|   amount|oldbalanceOrg|newbalanceOrig|isFraud|
+--------+---------+-------------+--------------+-------+
| PAYMENT|  9839.64|     170136.0|     160296.36|      0|
| PAYMENT|  1864.28|      21249.0|      19384.72|      0|
|TRANSFER|    181.0|        181.0|           0.0|      1|
|CASH_OUT|    181.0|        181.0|           0.0|      1|
| PAYMENT| 11668.14|      41554.0|      29885.86|      0|
| PAYMENT|  7817.71|      53860.0|      46042.29|      0|
| PAYMENT|  7107.77|     183195.0|     176087.23|      0|
| PAYMENT|  7861.64|    176087.23|     168225.59|      0|
| PAYMENT|  4024.36|       2671.0|           0.0|      0|
|   DEBIT|  5337.77|      41720.0|      36382.23|      0|
|   DEBIT|  9644.94|       4465.0|           0.0|      0|
| PAYMENT|  3099.97|      20771.0|      17671.03|      0|
| PAYMENT|  2560.74|       5070.0|       2509.26|      0|
| PAYMENT| 11633.76|      10127.0|           0.0|      0|
| PAYMENT|  40

In [6]:
# split the dataset into test (30%) and train (70%) dataset
(test, train) = df1.randomSplit([0.3, 0.7], seed = 9)

# Print the dataset
print(f'Train data set size: {train.count()} records')
print(f'Test data set size: {test.count()} records')

Train data set size: 4454447 records
Test data set size: 1908173 records


In [7]:
train.printSchema()
test.printSchema()

root
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- isFraud: integer (nullable = true)

root
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- isFraud: integer (nullable = true)



In [8]:
# Checking data type
catCols = [x for (x, dataType) in train.dtypes if dataType == 'string']
numCols = [x for (x, dataType) in train.dtypes if ((dataType == 'double') & (x != 'isFound'))]

print(catCols)
print(numCols)

['type']
['amount', 'oldbalanceOrg', 'newbalanceOrig']


In [9]:
# Determine distinct type of the 'type' features
train.agg(F.countDistinct('type')).show()

+-----------+
|count(type)|
+-----------+
|          5|
+-----------+



#### The value 5 indicates there are five different type of financial transaction. -> categorical feature

In [10]:
# find how many records for each type
train.groupBy('type').count().show()

+--------+-------+
|    type|  count|
+--------+-------+
|TRANSFER| 373211|
| CASH_IN| 979110|
|CASH_OUT|1566678|
| PAYMENT|1506427|
|   DEBIT|  29021|
+--------+-------+



In [11]:
# In regression, we need numerical values. Need to transform the category type (String) to numeric data type.
# One way to do that is to use one-hot-encoding
from pyspark.ml.feature import (
    OneHotEncoder,
    StringIndexer
)

stringIndexer = [
    StringIndexer(inputCol = x, outputCol = x + '_StringIndexer', handleInvalid = 'skip')
    for x in catCols
]

oneHotEncoder = [
    OneHotEncoder (
        inputCols = [f'{x}_StringIndexer' for x in catCols],
        outputCols = [f'{x}_OneHotEncoder' for x in catCols]
    )
]


In [12]:
# In PySpark, we need to transform (put) all features in to a vector and send to the machine learning model.
# We can use vector assemble to do this transformation.
from pyspark.ml.feature import VectorAssembler

In [13]:
assemblerInput = [x for x in numCols]
assemblerInput += [f'{x}_OneHotEncoder' for x in catCols]

In [14]:
assemblerInput

['amount', 'oldbalanceOrg', 'newbalanceOrig', 'type_OneHotEncoder']

In [15]:
vectorAssembler = VectorAssembler (
    inputCols = assemblerInput, 
    outputCol = 'VectorAssembler_features' 
)

In [16]:
stages = []
stages += stringIndexer
stages += oneHotEncoder
stages += [vectorAssembler]

In [17]:
stages

[StringIndexer_5dabf2c3a1d1,
 OneHotEncoder_a771b3029d4e,
 VectorAssembler_8699dfeae196]

In [18]:
from pyspark.ml import Pipeline

pipeline = Pipeline().setStages(stages)
model = pipeline.fit(train)

ppDf = model.transform(test)

In [19]:
ppDf.select(
    'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'VectorAssembler_features'
).show(200, truncate = False)

+-------+-------+-------------+--------------+-----------------------------------------------------+
|type   |amount |oldbalanceOrg|newbalanceOrig|VectorAssembler_features                             |
+-------+-------+-------------+--------------+-----------------------------------------------------+
|CASH_IN|14.4   |1.143460813E7|1.143462253E7 |[14.4,1.143460813E7,1.143462253E7,0.0,0.0,1.0,0.0]   |
|CASH_IN|14.54  |3347286.5    |3347301.03    |[14.54,3347286.5,3347301.03,0.0,0.0,1.0,0.0]         |
|CASH_IN|21.57  |104362.0     |104383.57     |[21.57,104362.0,104383.57,0.0,0.0,1.0,0.0]           |
|CASH_IN|22.67  |405940.0     |405962.67     |[22.67,405940.0,405962.67,0.0,0.0,1.0,0.0]           |
|CASH_IN|35.47  |3796691.21   |3796726.68    |[35.47,3796691.21,3796726.68,0.0,0.0,1.0,0.0]        |
|CASH_IN|45.02  |2745236.45   |2745281.46    |[45.02,2745236.45,2745281.46,0.0,0.0,1.0,0.0]        |
|CASH_IN|45.47  |4634070.59   |4634116.06    |[45.47,4634070.59,4634116.06,0.0,0.0,1.0,0.0]

#### Logistic Regression

In [20]:
# Import logistic Regression model
from pyspark.ml.classification import LogisticRegression

In [21]:
data = ppDf.select (
    F.col('vectorAssembler_features').alias('features'),
    F.col('isFraud').alias('label')
)

In [22]:
data.show(200, truncate=False)

+-----------------------------------------------------+-----+
|features                                             |label|
+-----------------------------------------------------+-----+
|[14.4,1.143460813E7,1.143462253E7,0.0,0.0,1.0,0.0]   |0    |
|[14.54,3347286.5,3347301.03,0.0,0.0,1.0,0.0]         |0    |
|[21.57,104362.0,104383.57,0.0,0.0,1.0,0.0]           |0    |
|[22.67,405940.0,405962.67,0.0,0.0,1.0,0.0]           |0    |
|[35.47,3796691.21,3796726.68,0.0,0.0,1.0,0.0]        |0    |
|[45.02,2745236.45,2745281.46,0.0,0.0,1.0,0.0]        |0    |
|[45.47,4634070.59,4634116.06,0.0,0.0,1.0,0.0]        |0    |
|[52.64,3468172.49,3468225.13,0.0,0.0,1.0,0.0]        |0    |
|[57.98,1290788.6,1290846.57,0.0,0.0,1.0,0.0]         |0    |
|[59.23,20841.0,20900.23,0.0,0.0,1.0,0.0]             |0    |
|[66.78,61853.0,61919.78,0.0,0.0,1.0,0.0]             |0    |
|[93.31,170084.31,170177.62,0.0,0.0,1.0,0.0]          |0    |
|[105.51,20594.0,20699.51,0.0,0.0,1.0,0.0]            |0    |
|[119.66

In [23]:
model = LogisticRegression().fit(data)

In [24]:
model.summary.areaUnderROC

0.9904370525762533

In [25]:
model.summary.pr.show()

+------------------+--------------------+
|            recall|           precision|
+------------------+--------------------+
|               0.0| 0.10844945462136632|
|0.8342031312725813| 0.10844945462136632|
| 0.895222802087515|0.058488735017179425|
|0.9409875551987154| 0.04105510211230602|
|0.9747089522280209| 0.03192174701884014|
|0.9935768767563228| 0.02604111866332779|
|0.9935768767563228|0.021703862849125267|
|0.9935768767563228| 0.01860356737498027|
|0.9935768767563228| 0.01627818263134356|
|0.9935768767563228| 0.01522795791546176|
|0.9935768767563228|0.013633133747927488|
|0.9935768767563228|0.012340446749102512|
|0.9939783219590526|0.011276021149370848|
|0.9939783219590526|0.010377200335289187|
|0.9939783219590526|0.009611129657089178|
|0.9939783219590526|0.008950163206732142|
|0.9939783219590526|0.008622851252333323|
|0.9939783219590526|0.008087089725541943|
|0.9939783219590526|0.007614314665551377|
|0.9947812123645122|0.007199219067762141|
+------------------+--------------